# ShopSphere - Phase 3: KPIs (Key Performance Indicators)

**Input:** `silver/silver.db` (clean tables from Phase 2). **Output:** one CSV per KPI in `kpi_outputs/`.

**How to read this notebook:** every KPI has the same 4 steps
1. **Markdown cell** - what the KPI means, how we calculate it, which assumptions we made
2. **SQL cell** - the query (plain SQL, run on SQLite) and its result table
3. **Check cell** - tests that stop the notebook if a number looks wrong, then saves the CSV
4. Next KPI

**Why SQL?** The clean data is already in a small database, so SQL (the standard language for questions on tables) is the shortest way to answer each KPI.

## Rules used by ALL KPIs (read once)

| Topic | Rule | Why |
|---|---|---|
| Revenue amount | `line_total` (quantity x price x (1 - discount%)) recomputed in Phase 2 | 50 stored line totals were wrong |
| Which orders are revenue | **Completed** and **Returned** orders. **Cancelled** and **Pending** are not revenue | money was not kept / not earned yet |
| Refunds | Only **Approved** returns reduce revenue. Rejected / Pending returns are ignored. Blank refund = 0. Refund is capped at the line total | a refund cannot be bigger than what was sold |
| Which month a refund belongs to | the month of the **order**, not the return date | 28 of 58 return dates are *before* the order date, so return dates cannot be trusted |
| What a "return" means | the whole order line is returned (the returns table has no quantity) | only information we have |
| Customers missing in the customer table | their 15 orders **stay** in revenue KPIs (1, 4). They are **left out** of customer KPIs (2, 3, 7) | revenue is real, but we do not know who the customer is |
| Dates like 03/04/2024 | read as MM/DD when both parts are 12 or less (flagged in `*_date_ambiguous` columns) | documented assumption; month KPIs (1, 4, 7) can shift slightly |

## Step 0 - Setup: open the database

In [1]:
import numpy as np
import pandas as pd
from kpi_lib import connect, q, save, check, get_params, summarize_recon, OUT_DIR

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

conn = connect()       # silver.db is only READ here, never changed

for t in ['customers', 'products', 'suppliers', 'orders', 'order_items', 'payments', 'returns', 'rejects']:
    print(f'{t:12s} rows = {conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]}')

customers    rows = 158
products     rows = 70
suppliers    rows = 18
orders       rows = 320
order_items  rows = 769
payments     rows = 317
returns      rows = 58
rejects      rows = 88


## Step 1 - Helper views (so each KPI query stays short)

A **view** is a saved query that behaves like a table. They are TEMP, so they disappear when the notebook closes and `silver.db` stays untouched.

| View | What it holds |
|---|---|
| `v_refund_per_item` | approved refund amount per order line |
| `v_sales_lines` | one row per **sold** line (Completed + Returned orders) with category, channel, month, refund and `net_revenue = line_total - refund` |
| `v_product_sales` | per product: units ordered, units returned, return rate |
| `v_customer_repeat` | per customer: first order, second order, days between (for KPI 7) |

In [2]:
views_sql = """-- Shared helper views (TEMP = they exist only while the connection is open, silver.db is not changed)

DROP VIEW IF EXISTS temp.v_refund_per_item;
DROP VIEW IF EXISTS temp.v_sales_lines;
DROP VIEW IF EXISTS temp.v_product_sales;
DROP VIEW IF EXISTS temp.v_customer_repeat;

-- 1) Approved refund per order line (blank refund = 0). Rejected / Pending returns are ignored.
CREATE TEMP VIEW v_refund_per_item AS
SELECT order_item_id,
       SUM(COALESCE(refund_amount, 0)) AS refund_amount
FROM returns
WHERE return_status = 'Approved'
GROUP BY order_item_id;

-- 2) One row per SOLD order line: only Completed and Returned orders count as revenue.
--    refund is capped at line_total, so a bad refund can never make revenue negative.
CREATE TEMP VIEW v_sales_lines AS
SELECT oi.order_item_id,
       oi.order_id,
       oi.product_id,
       oi.quantity,
       oi.line_total,
       o.customer_id,
       o.customer_orphan,
       o.order_date,
       o.order_date_ambiguous,
       SUBSTR(o.order_date, 1, 7)                            AS order_month,
       o.order_status,
       o.channel,
       p.category,
       p.supplier_id,
       CASE WHEN r.order_item_id IS NULL THEN 0 ELSE 1 END   AS is_returned,
       MIN(COALESCE(r.refund_amount, 0), oi.line_total)      AS refund,
       oi.line_total - MIN(COALESCE(r.refund_amount, 0), oi.line_total) AS net_revenue
FROM order_items oi
JOIN orders   o ON o.order_id   = oi.order_id
JOIN products p ON p.product_id = oi.product_id
LEFT JOIN v_refund_per_item r ON r.order_item_id = oi.order_item_id
WHERE o.order_status IN ('Completed', 'Returned');

-- 3) One row per product: units ordered, units returned, return rate (used by KPI 5 and KPI 6)
CREATE TEMP VIEW v_product_sales AS
SELECT product_id,
       SUM(quantity)                                              AS ordered_qty,
       SUM(CASE WHEN is_returned = 1 THEN quantity ELSE 0 END)    AS returned_qty,
       1.0 * SUM(CASE WHEN is_returned = 1 THEN quantity ELSE 0 END) / SUM(quantity) AS return_rate,
       SUM(line_total)                                            AS gross_revenue,
       SUM(net_revenue)                                           AS net_revenue
FROM v_sales_lines
GROUP BY product_id;

-- 4) One row per customer with at least one Completed order: first order, second order, days between (KPI 7)
CREATE TEMP VIEW v_customer_repeat AS
WITH seq AS (
    SELECT customer_id,
           order_date,
           order_date_ambiguous,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS order_no,
           LEAD(order_date)           OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date,
           LEAD(order_date_ambiguous) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_ambiguous
    FROM orders
    WHERE order_status = 'Completed'
      AND customer_orphan = 0
)
SELECT customer_id,
       order_date      AS first_order_date,
       next_order_date AS second_order_date,
       CASE WHEN next_order_date IS NULL THEN NULL
            ELSE CAST(JULIANDAY(next_order_date) - JULIANDAY(order_date) AS INTEGER) END AS days_to_second_order,
       MAX(order_date_ambiguous, COALESCE(next_ambiguous, 0)) AS uses_ambiguous_date
FROM seq
WHERE order_no = 1;
"""

conn.executescript(views_sql)

print('sold order lines :', q(conn, 'SELECT COUNT(*) AS n FROM v_sales_lines').n[0])
print('approved refunds :', q(conn, 'SELECT COUNT(*) AS n FROM v_refund_per_item').n[0])

sold order lines : 540
approved refunds : 28


### Data issue found while building the views

Some returns point to lines of **Cancelled / Pending** orders. A line that was never sold cannot be returned, so these returns are **ignored** (they are not in `v_sales_lines`). The table shows how many returns exist per order status:

In [3]:
q(conn, """
SELECT o.order_status, COUNT(*) AS returns_rows
FROM returns r
JOIN order_items oi ON oi.order_item_id = r.order_item_id
JOIN orders o       ON o.order_id = oi.order_id
GROUP BY o.order_status
""")

,order_status,returns_rows
0,Cancelled,7
1,Completed,32
2,Pending,8
3,Returned,11


## Step 2 - Settings (assumptions in one place)

Kept in `kpi_lib.py` so the notebook and the pipeline always use the same values.
`min_units` (KPI 5) is calculated from the data in KPI 5 below.

In [4]:
PARAMS = get_params(conn)
PARAMS

{'w_revenue': 0.4,
 'w_return': 0.3,
 'w_rating': 0.3,
 'repeat_window_days': 90,
 'pay_tolerance': 0.01,
 'min_units': 18}

---
## KPI 1 - Net revenue by product category, per month

**Question:** how much money did each category earn each month, after refunds?

**How:** take every sold line (`v_sales_lines`), group by order month + category, and add up
`gross revenue` (line totals), `approved refunds`, and `net revenue = gross - refunds`.

**Assumptions:** see the rules table (Completed + Returned orders, approved refunds only, refund counted in the order month).
Orders of customers missing from the customer table are **included**.

In [5]:
sql = """-- KPI 1: net revenue by product category per month
SELECT order_month,
       category,
       COUNT(DISTINCT order_id)      AS orders,
       SUM(quantity)                 AS units,
       ROUND(SUM(line_total), 2)     AS gross_revenue,
       ROUND(SUM(refund), 2)         AS approved_refunds,
       ROUND(SUM(net_revenue), 2)    AS net_revenue
FROM v_sales_lines
GROUP BY order_month, category
ORDER BY order_month, category;
"""

kpi1 = q(conn, sql, PARAMS)
print(kpi1.shape)
kpi1.head(15)

(118, 7)


,order_month,category,orders,units,gross_revenue,approved_refunds,net_revenue
0,2020-10,Grocery,1,2,"45,144.50",0.00,"45,144.50"
1,2020-12,Home & Kitchen,1,1,"21,122.18",0.00,"21,122.18"
2,2021-09,Grocery,1,3,"13,037.07",0.00,"13,037.07"
3,2021-09,Home & Kitchen,1,4,"56,241.65",0.00,"56,241.65"
4,2021-12,Electronics,1,1,"16,084.14",0.00,"16,084.14"
5,2021-12,Grocery,1,5,"88,376.90",0.00,"88,376.90"
6,2022-05,Fashion,1,5,"9,607.25",0.00,"9,607.25"
7,2022-05,Grocery,1,1,"17,582.77",0.00,"17,582.77"
8,2022-05,Sports,1,4,"81,924.16",0.00,"81,924.16"
9,2024-01,Electronics,3,8,"152,376.05",0.00,"152,376.05"


**Check KPI 1** - rebuild the same total in pandas (a different way) and compare. Then save.

In [6]:
# independent calculation with pandas
oi = q(conn, "SELECT order_item_id, order_id, line_total FROM order_items")
od = q(conn, "SELECT order_id, order_status FROM orders")
rf = q(conn, "SELECT order_item_id, refund_amount FROM returns WHERE return_status = 'Approved'")

m = oi.merge(od, on='order_id')
m = m[m.order_status.isin(['Completed', 'Returned'])].copy()
m['refund'] = m.order_item_id.map(rf.groupby('order_item_id').refund_amount.sum()).fillna(0)
m['refund'] = m[['refund', 'line_total']].min(axis=1)
pandas_net = (m.line_total - m.refund).sum()

check('KPI1 total net revenue = independent pandas total', abs(kpi1.net_revenue.sum() - pandas_net) < 1)
check('KPI1 gross - refunds = net on every row', np.allclose(kpi1.gross_revenue - kpi1.approved_refunds, kpi1.net_revenue, atol=0.02))
check('KPI1 no empty month or category', kpi1[['order_month', 'category']].notna().all().all())

print('\nTotals by category:')
display(kpi1.groupby('category')[['gross_revenue', 'approved_refunds', 'net_revenue']].sum().sort_values('net_revenue', ascending=False))
save(kpi1, 'kpi_01_net_revenue_by_category_month')

PASS  KPI1 total net revenue = independent pandas total
PASS  KPI1 gross - refunds = net on every row
PASS  KPI1 no empty month or category

Totals by category:


,gross_revenue,approved_refunds,net_revenue
category,,,
Grocery,"5,283,076.68","120,071.12","5,163,005.56"
Home & Kitchen,"5,044,013.53","209,868.31","4,834,145.22"
Fashion,"4,238,682.36","210,360.31","4,028,322.05"
Sports,"2,895,583.77","199,788.28","2,695,795.49"
Electronics,"2,629,489.72","60,115.27","2,569,374.45"


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_01_net_revenue_by_category_month.csv')

---
## KPI 2 - Top 10 customers by net lifetime value (LTV)

**Question:** which 10 customers brought the most money over their whole history?

**How:** for each customer, add up the net revenue (line total minus approved refunds) of their **Completed** orders. Sort high to low, keep 10.

**Assumptions:** only **Completed** orders (Returned-status orders are not counted here, because LTV is about money kept).
Orders whose customer is missing in the customer table cannot be named, so they are **excluded**.

In [7]:
sql = """-- KPI 2: top 10 customers by net lifetime value (Completed orders only, orphan customers excluded)
SELECT c.customer_id,
       c.first_name || ' ' || c.last_name AS customer_name,
       c.customer_segment                 AS segment,
       c.city,
       COUNT(DISTINCT s.order_id)         AS completed_orders,
       ROUND(SUM(s.line_total), 2)        AS gross_value,
       ROUND(SUM(s.refund), 2)            AS approved_refunds,
       ROUND(SUM(s.net_revenue), 2)       AS net_lifetime_value
FROM v_sales_lines s
JOIN customers c ON c.customer_id = s.customer_id
WHERE s.order_status = 'Completed'
  AND s.customer_orphan = 0
GROUP BY c.customer_id
ORDER BY net_lifetime_value DESC, c.customer_id
LIMIT 10;
"""

kpi2 = q(conn, sql, PARAMS)
kpi2

,customer_id,customer_name,segment,city,completed_orders,gross_value,approved_refunds,net_lifetime_value
0,C0118,Utkarsh Bhat,Gold,Hyderabad,3,"532,447.01",0.00,"532,447.01"
1,C0107,Eesha Sethi,Silver,Kolkata,2,"463,604.51",0.00,"463,604.51"
2,C0061,Yadavi Bhakta,Platinum,Chennai,2,"341,730.61",0.00,"341,730.61"
3,C0148,Ucchal Thaker,Bronze,Jaipur,3,"329,199.64",0.00,"329,199.64"
4,C0142,Zansi Prasad,Platinum,Ahmedabad,4,"328,510.36",0.00,"328,510.36"
5,C0001,Isaac Bakshi,Bronze,Bangalore,4,"377,880.04","75,816.05","302,063.99"
6,C0085,Chakrika Koshy,Silver,Chennai,3,"291,047.36",0.00,"291,047.36"
7,C0078,Umang Mital,Platinum,Delhi,3,"284,225.67",0.00,"284,225.67"
8,C0033,Hamsini Rao,Gold,Lucknow,2,"268,013.97",0.00,"268,013.97"
9,C0129,Siddharth Bains,Platinum,Delhi,3,"284,905.07","19,128.89","265,776.18"


**Check KPI 2** - same ranking with pandas, then save.

In [8]:
oi = q(conn, "SELECT order_item_id, order_id, line_total FROM order_items")
od = q(conn, "SELECT order_id, customer_id, order_status, customer_orphan FROM orders")
rf = q(conn, "SELECT order_item_id, refund_amount FROM returns WHERE return_status = 'Approved'")

m = oi.merge(od, on='order_id')
m = m[(m.order_status == 'Completed') & (m.customer_orphan == 0)].copy()
m['refund'] = m.order_item_id.map(rf.groupby('order_item_id').refund_amount.sum()).fillna(0)
m['refund'] = m[['refund', 'line_total']].min(axis=1)
ltv = (m.line_total - m.refund).groupby(m.customer_id).sum().sort_values(ascending=False).head(10)

check('KPI2 exactly 10 customers', len(kpi2) == 10)
check('KPI2 same 10 customers and values as pandas', list(ltv.index) == list(kpi2.customer_id) and np.allclose(ltv.values, kpi2.net_lifetime_value, atol=0.01))
check('KPI2 sorted high to low', kpi2.net_lifetime_value.is_monotonic_decreasing)
print(f"\nTop customer: {kpi2.customer_name[0]} ({kpi2.customer_id[0]}) with net LTV {kpi2.net_lifetime_value[0]:,.2f}")
save(kpi2, 'kpi_02_top10_customers_ltv')

PASS  KPI2 exactly 10 customers
PASS  KPI2 same 10 customers and values as pandas
PASS  KPI2 sorted high to low

Top customer: Utkarsh Bhat (C0118) with net LTV 532,447.01


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_02_top10_customers_ltv.csv')

---
## KPI 3 - Customer segment profile

**Question:** how many customers does each segment (Gold, Silver, Platinum, Bronze) have, and how big is their average order?

**How:** `customers` = all customers in the segment. **AOV (Average Order Value)** = total value of Completed orders / number of Completed orders.
We also show AOV after approved returns.

**Assumptions:** only Completed orders; customers with no Completed order stay in the customer count but add nothing to AOV.

In [9]:
sql = """-- KPI 3: customer segment profile = customers + average order value (AOV)
WITH order_value AS (
    SELECT order_id,
           SUM(line_total)  AS gross_value,
           SUM(net_revenue) AS net_value
    FROM v_sales_lines
    GROUP BY order_id
)
SELECT COALESCE(c.customer_segment, 'Unknown')             AS segment,
       COUNT(DISTINCT c.customer_id)                       AS customers,
       COUNT(DISTINCT o.customer_id)                       AS customers_with_completed_order,
       COUNT(o.order_id)                                   AS completed_orders,
       ROUND(SUM(COALESCE(ov.gross_value, 0)), 2)          AS total_order_value,
       ROUND(SUM(COALESCE(ov.gross_value, 0)) / NULLIF(COUNT(o.order_id), 0), 2) AS avg_order_value,
       ROUND(SUM(COALESCE(ov.net_value, 0))   / NULLIF(COUNT(o.order_id), 0), 2) AS avg_order_value_after_returns
FROM customers c
LEFT JOIN orders o       ON o.customer_id = c.customer_id AND o.order_status = 'Completed'
LEFT JOIN order_value ov ON ov.order_id   = o.order_id
GROUP BY COALESCE(c.customer_segment, 'Unknown')
ORDER BY avg_order_value DESC;
"""

kpi3 = q(conn, sql, PARAMS)
kpi3

,segment,customers,customers_with_completed_order,completed_orders,total_order_value,avg_order_value,avg_order_value_after_returns
0,Gold,38,21,33,"3,195,581.49","96,835.80","93,823.28"
1,Silver,40,25,36,"3,259,623.23","90,545.09","86,200.89"
2,Platinum,44,30,52,"4,053,544.73","77,952.78","76,512.09"
3,Bronze,36,22,36,"2,788,736.47","77,464.90","72,698.72"


**Check KPI 3**

In [10]:
total_customers = conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
check('KPI3 customer counts add up to the customers table', kpi3.customers.sum() == total_customers)
check('KPI3 AOV = total order value / completed orders',
      np.allclose(kpi3.total_order_value / kpi3.completed_orders, kpi3.avg_order_value, atol=0.02))
check('KPI3 AOV after returns is never above AOV', (kpi3.avg_order_value_after_returns <= kpi3.avg_order_value + 0.01).all())
print(f"\nHighest AOV: {kpi3.segment[0]} ({kpi3.avg_order_value[0]:,.2f})")
save(kpi3, 'kpi_03_segment_profile')

PASS  KPI3 customer counts add up to the customers table
PASS  KPI3 AOV = total order value / completed orders
PASS  KPI3 AOV after returns is never above AOV

Highest AOV: Gold (96,835.80)


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_03_segment_profile.csv')

---
## KPI 4 - Month-over-month (MoM) growth by channel

**Question:** for Web, App and Store, how did the number of orders and the net revenue change compared with the month before?

**How:** build a full calendar (every month x every channel, so a month with no orders shows 0), then use the window function `LAG()` to read the previous month and calculate
`growth % = (this month - last month) / last month x 100`.

**Assumptions:** orders = Completed + Returned orders. Growth is **empty (NULL)** when last month was 0 (cannot divide by 0) and for the first month of each channel.
The data has only a few orders per month, so single-order changes look big; read the yearly summary below for the bigger picture.

In [11]:
sql = """-- KPI 4: month-over-month (MoM) growth of order count and net revenue, per channel
WITH RECURSIVE months(month_start) AS (               -- every calendar month from first to last order
    SELECT DATE(MIN(order_date), 'start of month') FROM orders
    UNION ALL
    SELECT DATE(month_start, '+1 month') FROM months
    WHERE month_start < (SELECT DATE(MAX(order_date), 'start of month') FROM orders)
),
grid AS (                                            -- every channel x every month (so empty months show as 0)
    SELECT ch.channel, SUBSTR(m.month_start, 1, 7) AS order_month
    FROM (SELECT DISTINCT channel FROM orders WHERE channel IS NOT NULL) ch
    CROSS JOIN months m
),
order_cnt AS (
    SELECT channel, SUBSTR(order_date, 1, 7) AS order_month, COUNT(*) AS orders
    FROM orders
    WHERE order_status IN ('Completed', 'Returned')
    GROUP BY channel, SUBSTR(order_date, 1, 7)
),
rev AS (
    SELECT channel, order_month, SUM(net_revenue) AS net_revenue
    FROM v_sales_lines
    GROUP BY channel, order_month
),
monthly AS (
    SELECT g.channel, g.order_month,
           COALESCE(oc.orders, 0)      AS orders,
           COALESCE(r.net_revenue, 0)  AS net_revenue
    FROM grid g
    LEFT JOIN order_cnt oc ON oc.channel = g.channel AND oc.order_month = g.order_month
    LEFT JOIN rev r        ON r.channel  = g.channel AND r.order_month  = g.order_month
),
with_prev AS (
    SELECT *,
           LAG(orders)      OVER (PARTITION BY channel ORDER BY order_month) AS prev_orders,
           LAG(net_revenue) OVER (PARTITION BY channel ORDER BY order_month) AS prev_net_revenue
    FROM monthly
)
SELECT channel,
       order_month,
       orders,
       prev_orders,
       ROUND(100.0 * (orders - prev_orders) / NULLIF(prev_orders, 0), 2)                 AS orders_mom_pct,
       ROUND(net_revenue, 2)                                                             AS net_revenue,
       ROUND(prev_net_revenue, 2)                                                        AS prev_net_revenue,
       ROUND(100.0 * (net_revenue - prev_net_revenue) / NULLIF(prev_net_revenue, 0), 2)  AS revenue_mom_pct
FROM with_prev
ORDER BY channel, order_month;
"""

kpi4 = q(conn, sql, PARAMS)
print(kpi4.shape)
kpi4[kpi4.orders > 0].head(15)

(189, 8)


,channel,order_month,orders,prev_orders,orders_mom_pct,net_revenue,prev_net_revenue,revenue_mom_pct
2,App,2020-12,1,0,NaN,"21,122.18",0.00,NaN
11,App,2021-09,1,0,NaN,"69,278.72",0.00,NaN
19,App,2022-05,1,0,NaN,"109,114.18",0.00,NaN
39,App,2024-01,7,0,NaN,"614,998.21",0.00,NaN
40,App,2024-02,3,7,-57.14,"250,522.00","614,998.21",-59.26
41,App,2024-03,5,3,66.67,"290,497.56","250,522.00",15.96
42,App,2024-04,4,5,-20.00,"372,671.90","290,497.56",28.29
43,App,2024-05,2,4,-50.00,"248,097.95","372,671.90",-33.43
44,App,2024-06,7,2,250.00,"444,413.41","248,097.95",79.13
45,App,2024-07,3,7,-57.14,"109,033.56","444,413.41",-75.47


**Check KPI 4**

In [12]:
n_orders = conn.execute("SELECT COUNT(*) FROM orders WHERE order_status IN ('Completed','Returned')").fetchone()[0]
check('KPI4 order counts add up to all revenue orders', kpi4.orders.sum() == n_orders)
check('KPI4 revenue total = KPI1 revenue total', abs(kpi4.net_revenue.sum() - kpi1.net_revenue.sum()) < 1)
first = kpi4.groupby('channel').head(1)
check('KPI4 first month of each channel has no growth value', first.orders_mom_pct.isna().all() and first.revenue_mom_pct.isna().all())
check('KPI4 same number of months for every channel', kpi4.groupby('channel').size().nunique() == 1)

print('\nYearly summary by channel (easier to read than single months):')
yearly = (kpi4.assign(year=kpi4.order_month.str[:4])
              .groupby(['year', 'channel'])[['orders', 'net_revenue']].sum().reset_index())
display(yearly.pivot(index='year', columns='channel', values='net_revenue'))
save(kpi4, 'kpi_04_channel_mom_growth')

PASS  KPI4 order counts add up to all revenue orders
PASS  KPI4 revenue total = KPI1 revenue total
PASS  KPI4 first month of each channel has no growth value
PASS  KPI4 same number of months for every channel

Yearly summary by channel (easier to read than single months):


channel,App,Store,Web
year,,,
2020,"21,122.18","45,144.50",0.00
2021,"69,278.72","104,461.04",0.00
2022,"109,114.18",0.00,0.00
2023,0.00,0.00,0.00
2024,"4,360,667.99","3,461,264.27","2,762,833.21"
2025,"3,003,857.42","2,832,680.74","2,520,218.52"


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_04_channel_mom_growth.csv')

---
## KPI 5 - Product return rate (top 10 and bottom 10)

**Question:** which products are returned most, and which least?

**How:** `return rate = returned units / ordered units` (a returned line = all units of that line, only Approved returns).
Ties are broken by higher ordered volume, then product id.

**Minimum volume:** a product sold 2 times with 1 return would show 50% - just noise. We drop the 25% least-sold products.
The limit comes from the data: the 25th percentile of units ordered per product (shown below).
**Note:** many products have 0 returns, so the bottom 10 are all 0% and are ordered by volume (more units sold with zero returns = better).

In [13]:
dist = q(conn, "SELECT ordered_qty FROM v_product_sales", PARAMS).ordered_qty
print('Units ordered per product - distribution:')
print(dist.describe().round(1).to_string())
print(f"\nMinimum volume used (25th percentile) = {PARAMS['min_units']} units")
print(f"Products kept: {(dist >= PARAMS['min_units']).sum()} of {len(dist)}")

Units ordered per product - distribution:
count   70.00
mean    23.80
std      9.40
min      6.00
25%     18.00
50%     25.00
75%     30.00
max     51.00

Minimum volume used (25th percentile) = 18 units
Products kept: 53 of 70


In [14]:
sql = """-- KPI 5: product return rate = returned units / ordered units. Top 10 and bottom 10, minimum volume applied.
WITH base AS (
    SELECT ps.product_id, p.product_name, p.category,
           ps.ordered_qty, ps.returned_qty, ps.return_rate
    FROM v_product_sales ps
    JOIN products p ON p.product_id = ps.product_id
    WHERE ps.ordered_qty >= :min_units
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY return_rate DESC, ordered_qty DESC, product_id) AS rank_top,
           ROW_NUMBER() OVER (ORDER BY return_rate ASC,  ordered_qty DESC, product_id) AS rank_bottom
    FROM base
)
SELECT 'Top 10 - highest return rate' AS list_name, rank_top AS rank_no,
       product_id, product_name, category, ordered_qty, returned_qty,
       ROUND(100.0 * return_rate, 2) AS return_rate_pct
FROM ranked WHERE rank_top <= 10
UNION ALL
SELECT 'Bottom 10 - lowest return rate', rank_bottom,
       product_id, product_name, category, ordered_qty, returned_qty,
       ROUND(100.0 * return_rate, 2)
FROM ranked WHERE rank_bottom <= 10
ORDER BY list_name DESC, rank_no;
"""

kpi5 = q(conn, sql, PARAMS)
kpi5

,list_name,rank_no,product_id,product_name,category,ordered_qty,returned_qty,return_rate_pct
0,Top 10 - highest return rate,1,P029,Furniture Item 29,Home & Kitchen,28,6,21.43
1,Top 10 - highest return rate,2,P052,Womenswear Item 52,Fashion,27,5,18.52
2,Top 10 - highest return rate,3,P057,Team Sports Item 57,Sports,29,5,17.24
3,Top 10 - highest return rate,4,P040,Footwear Item 40,Fashion,31,5,16.13
4,Top 10 - highest return rate,5,P009,Furniture Item 9,Home & Kitchen,25,4,16.00
5,Top 10 - highest return rate,6,P034,Fitness Item 34,Sports,25,4,16.00
6,Top 10 - highest return rate,7,P030,Furniture Item 30,Home & Kitchen,26,4,15.38
7,Top 10 - highest return rate,8,P021,Fitness Item 21,Sports,35,5,14.29
8,Top 10 - highest return rate,9,P035,Cookware Item 35,Home & Kitchen,28,4,14.29
9,Top 10 - highest return rate,10,P058,Snacks Item 58,Grocery,28,3,10.71


**Check KPI 5**

In [15]:
check('KPI5 20 rows (10 top + 10 bottom)', len(kpi5) == 20)
check('KPI5 every product has at least the minimum volume', (kpi5.ordered_qty >= PARAMS['min_units']).all())
check('KPI5 rate is between 0 and 100', kpi5.return_rate_pct.between(0, 100).all())
check('KPI5 rate = returned / ordered', np.allclose(100 * kpi5.returned_qty / kpi5.ordered_qty, kpi5.return_rate_pct, atol=0.01))
top = kpi5[kpi5.list_name.str.startswith('Top')]
check('KPI5 top list sorted high to low', top.return_rate_pct.is_monotonic_decreasing)
save(kpi5, 'kpi_05_product_return_rate')

PASS  KPI5 20 rows (10 top + 10 bottom)
PASS  KPI5 every product has at least the minimum volume
PASS  KPI5 rate is between 0 and 100
PASS  KPI5 rate = returned / ordered
PASS  KPI5 top list sorted high to low


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_05_product_return_rate.csv')

---
## KPI 6 - Supplier scorecard

**Question:** which suppliers are best, using revenue, returns and rating together?

**How:**
1. For each supplier: net revenue (all their products), average return rate of their products, rating (0 to 5).
2. Put each of the three on a 0-1 scale with **min-max scaling**: `(value - lowest) / (highest - lowest)`.
3. Return rate is **inverted** (`1 - scaled`) because a low return rate is good.
4. `score = 100 x (0.40 x revenue + 0.30 x return + 0.30 x rating)`.

**Assumptions:**
- The weights 40 / 30 / 30 are **our assumption** (the case does not give them). A sensitivity test with equal weights is shown below.
- Four supplier ratings were on a 0-10 scale and were halved in Phase 2.
- Products with no known supplier go to an **Unknown** row, shown at the bottom but **not ranked**.

In [16]:
sql = """-- KPI 6: supplier scorecard. Score = weights x (revenue, 1 - return rate, rating), each scaled 0..1 (min-max)
WITH supplier_sales AS (
    SELECT COALESCE(p.supplier_id, 'UNKNOWN') AS supplier_id,
           COUNT(*)                           AS products_sold,
           SUM(ps.net_revenue)                AS net_revenue,
           AVG(ps.return_rate)                AS avg_return_rate
    FROM v_product_sales ps
    JOIN products p ON p.product_id = ps.product_id
    GROUP BY COALESCE(p.supplier_id, 'UNKNOWN')
),
joined AS (
    SELECT ss.supplier_id,
           COALESCE(s.supplier_name, 'Unknown supplier') AS supplier_name,
           s.country, s.rating,
           ss.products_sold, ss.net_revenue, ss.avg_return_rate
    FROM supplier_sales ss
    LEFT JOIN suppliers s ON s.supplier_id = ss.supplier_id
),
bounds AS (                                          -- min and max over the real suppliers only
    SELECT MIN(net_revenue) AS rev_min, MAX(net_revenue) AS rev_max,
           MIN(avg_return_rate) AS ret_min, MAX(avg_return_rate) AS ret_max,
           MIN(rating) AS rat_min, MAX(rating) AS rat_max
    FROM joined WHERE supplier_id <> 'UNKNOWN'
),
scaled AS (
    SELECT j.*,
           (j.net_revenue     - b.rev_min) / NULLIF(b.rev_max - b.rev_min, 0)       AS revenue_scaled,
           1 - (j.avg_return_rate - b.ret_min) / NULLIF(b.ret_max - b.ret_min, 0)   AS return_scaled,
           (j.rating          - b.rat_min) / NULLIF(b.rat_max - b.rat_min, 0)       AS rating_scaled
    FROM joined j CROSS JOIN bounds b
    WHERE j.supplier_id <> 'UNKNOWN'
),
scored AS (
    SELECT *,
           100 * (:w_revenue * revenue_scaled + :w_return * return_scaled + :w_rating * rating_scaled) AS score
    FROM scaled
)
SELECT * FROM (
    SELECT RANK() OVER (ORDER BY score DESC) AS rank_no,
           supplier_id, supplier_name, country, products_sold,
           ROUND(net_revenue, 2)          AS net_revenue,
           ROUND(100 * avg_return_rate, 2) AS avg_return_rate_pct,
           rating,
           ROUND(revenue_scaled, 3) AS revenue_scaled,
           ROUND(return_scaled, 3)  AS return_scaled,
           ROUND(rating_scaled, 3)  AS rating_scaled,
           ROUND(score, 2)          AS score
    FROM scored
    UNION ALL                                        -- products with no known supplier: shown, but not ranked
    SELECT NULL, supplier_id, supplier_name, NULL, products_sold,
           ROUND(net_revenue, 2), ROUND(100 * avg_return_rate, 2),
           NULL, NULL, NULL, NULL, NULL
    FROM joined WHERE supplier_id = 'UNKNOWN'
)
ORDER BY rank_no IS NULL, rank_no;
"""

kpi6 = q(conn, sql, PARAMS)
kpi6['rank_no'] = kpi6['rank_no'].astype('Int64')
kpi6

,rank_no,supplier_id,supplier_name,country,products_sold,net_revenue,avg_return_rate_pct,rating,revenue_scaled,return_scaled,rating_scaled,score
0,1,S006,"Lalla, Ganesh and Oommen",Germany,9,"2,469,077.71",2.29,3.75,1.00,0.86,0.63,84.68
1,2,S007,"Biswas, Karpe and Radhakrishnan",None,6,"1,861,322.65",0.00,4.20,0.73,1.00,0.77,82.08
2,3,S015,"Chaudhari, Sandal and Desai",India,4,"1,821,770.44",2.27,4.10,0.71,0.86,0.73,76.23
3,4,S004,"Sinha, Subramaniam and Garde",China,4,"1,612,831.39",6.52,4.70,0.62,0.59,0.91,69.81
4,5,S005,Dhar-Gopal,Germany,2,"910,835.69",0.00,4.50,0.30,1.00,0.85,67.73
5,6,S018,"Amble, Virk and Dalal",Vietnam,4,"615,765.99",2.27,5.00,0.17,0.86,1.00,62.61
6,7,S001,"Baral, Din and Bhalla",None,3,"1,336,183.94",7.10,4.30,0.49,0.56,0.79,60.26
7,8,S009,"Saini, Doctor and Char",Germany,2,"657,783.34",0.00,4.00,0.19,1.00,0.71,58.80
8,9,S008,Babu and Sons,India,2,"611,245.71",0.00,3.70,0.17,1.00,0.62,55.32
9,10,S010,"Barad, Wali and Thakur",Germany,3,"451,335.48",4.76,4.90,0.10,0.70,0.97,54.12


**Check KPI 6** - weights, score range, and what happens if the weights change.

In [17]:
check('KPI6 weights add up to 1', abs(PARAMS['w_revenue'] + PARAMS['w_return'] + PARAMS['w_rating'] - 1) < 1e-9)
ranked = kpi6[kpi6.rank_no.notna()].copy()
check('KPI6 scores are between 0 and 100', ranked.score.between(0, 100).all())
check('KPI6 Unknown supplier row is not ranked', kpi6[kpi6.supplier_id == 'UNKNOWN'].rank_no.isna().all())
check('KPI6 ranked rows are sorted by score', ranked.score.is_monotonic_decreasing)

suppliers_total = conn.execute("SELECT COUNT(*) FROM suppliers").fetchone()[0]
print(f'Suppliers in table: {suppliers_total}, ranked here: {len(ranked)} (the others sold nothing)')

# sensitivity: do the top 3 change with equal weights?
ranked['equal_weight_score'] = 100 * (ranked.revenue_scaled + ranked.return_scaled + ranked.rating_scaled) / 3
print('\nTop 3 with 40/30/30 :', list(ranked.sort_values('score', ascending=False).supplier_id.head(3)))
print('Top 3 with equal weights:', list(ranked.sort_values('equal_weight_score', ascending=False).supplier_id.head(3)))
save(kpi6, 'kpi_06_supplier_scorecard')

PASS  KPI6 weights add up to 1
PASS  KPI6 scores are between 0 and 100
PASS  KPI6 Unknown supplier row is not ranked
PASS  KPI6 ranked rows are sorted by score


Suppliers in table: 18, ranked here: 18 (the others sold nothing)

Top 3 with 40/30/30 : ['S006', 'S007', 'S015']
Top 3 with equal weights: ['S007', 'S006', 'S015']


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_06_supplier_scorecard.csv')

---
## KPI 7 - Repeat-purchase rate (within 90 days)

**Question:** what share of customers buy a second time soon after their first purchase?

**How:** for every customer with at least 1 **Completed** order, sort their Completed orders by date.
`ROW_NUMBER()` marks the first order and `LEAD()` reads the next order's date.
Repeat customer = second order **within 90 days** (same day counts).

**Rate = repeat customers / customers with at least one Completed order.**

**Assumptions:** 90 days is the window given in the task; Completed orders only; orphan customers excluded.
Some dates are ambiguous (MM/DD vs DD/MM), so we also report how many repeat customers depend on such a date.

In [18]:
views_note = "v_customer_repeat (created in Step 1) holds first order, second order and days between."
print(views_note)

v_customer_repeat (created in Step 1) holds first order, second order and days between.


In [19]:
sql = """-- KPI 7: repeat-purchase rate = customers whose 2nd Completed order is within N days of their 1st
SELECT COUNT(*) AS customers_with_completed_order,
       SUM(CASE WHEN days_to_second_order <= :repeat_window_days THEN 1 ELSE 0 END) AS repeat_customers,
       ROUND(100.0 * SUM(CASE WHEN days_to_second_order <= :repeat_window_days THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_rate_pct,
       SUM(CASE WHEN days_to_second_order <= :repeat_window_days AND uses_ambiguous_date = 1 THEN 1 ELSE 0 END) AS repeat_customers_with_ambiguous_dates
FROM v_customer_repeat;
"""

kpi7 = q(conn, sql, PARAMS)
kpi7

,customers_with_completed_order,repeat_customers,repeat_rate_pct,repeat_customers_with_ambiguous_dates
0,98,8,8.16,3


Per-customer detail (one row per customer, used to explain the rate):

In [20]:
sql = """-- KPI 7 detail: one row per customer (used to explain the rate)
SELECT customer_id, first_order_date, second_order_date, days_to_second_order,
       CASE WHEN days_to_second_order <= :repeat_window_days THEN 1 ELSE 0 END AS is_repeat_within_window,
       uses_ambiguous_date
FROM v_customer_repeat
ORDER BY customer_id;
"""

kpi7_detail = q(conn, sql, PARAMS)
kpi7_detail.head(10)

,customer_id,first_order_date,second_order_date,days_to_second_order,is_repeat_within_window,uses_ambiguous_date
0,C0001,2024-01-01,2024-03-28,87,1,0
1,C0002,2024-09-11,None,<NA>,0,1
2,C0004,2024-03-01,2024-03-09,8,1,0
3,C0005,2025-01-05,None,<NA>,0,1
4,C0006,2025-02-26,2025-08-21,176,0,0
5,C0007,2025-03-02,2025-09-15,197,0,0
6,C0010,2025-03-18,2025-08-06,141,0,0
7,C0011,2024-08-13,None,<NA>,0,0
8,C0013,2024-07-13,None,<NA>,0,0
9,C0014,2024-02-15,None,<NA>,0,0


**Check KPI 7** - recompute with pandas, then save.

In [21]:
oo = q(conn, "SELECT customer_id, order_id, order_date FROM orders WHERE order_status = 'Completed' AND customer_orphan = 0")
oo['order_date'] = pd.to_datetime(oo.order_date)
oo = oo.sort_values(['customer_id', 'order_date', 'order_id'])
gap = oo.groupby('customer_id').order_date.apply(lambda s: float((s.iloc[1] - s.iloc[0]).days) if len(s) > 1 else np.nan)

check('KPI7 customers with a completed order = pandas count', len(gap) == kpi7.customers_with_completed_order[0])
check('KPI7 repeat customers = pandas count', int((gap <= PARAMS['repeat_window_days']).sum()) == kpi7.repeat_customers[0])
check('KPI7 detail rows = customers in summary', len(kpi7_detail) == kpi7.customers_with_completed_order[0])
check('KPI7 rate is between 0 and 100', 0 <= kpi7.repeat_rate_pct[0] <= 100)

r = kpi7.iloc[0]
print(f"\nRepeat rate = {int(r.repeat_customers)} / {int(r.customers_with_completed_order)} = {r.repeat_rate_pct}%")
print(f"Repeat customers that depend on an ambiguous date: {int(r.repeat_customers_with_ambiguous_dates)}")
save(kpi7, 'kpi_07_repeat_purchase_rate')
save(kpi7_detail, 'kpi_07_repeat_purchase_detail')

PASS  KPI7 customers with a completed order = pandas count
PASS  KPI7 repeat customers = pandas count
PASS  KPI7 detail rows = customers in summary
PASS  KPI7 rate is between 0 and 100

Repeat rate = 8 / 98 = 8.16%
Repeat customers that depend on an ambiguous date: 3


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_07_repeat_purchase_detail.csv')

---
## KPI 8 - Payment reconciliation

**Question:** does the money received match what each order should have cost?

**How:** per order, `expected = sum of line totals`, `paid = sum of Success payments`. Then one status per order:

| Status | Meaning |
|---|---|
| Matched | paid is within 1% of expected |
| Underpaid / Overpaid | paid is more than 1% below / above expected |
| No successful payment | Completed or Returned order but no Success payment |
| Pending - not paid yet (OK) | Pending order, nothing paid, which is normal |
| Payment on cancelled order | Cancelled order that still has a Success payment (problem) |
| Cancelled - no payment (OK) | normal |
| Order has no items | the order has no lines, so no expected amount |

Extra flags per order: `duplicate_success_payment` (more than one Success payment with the same amount) and `conflicting_success_amounts` (Success payments with different amounts).

**Assumptions:** 1% tolerance; Failed, Pending and Refunded payments are not counted as money received; $-marked amounts were **not** converted (Phase 2 showed they match order totals as plain numbers).
The 8 payments whose order does not exist (quarantined in Phase 2) are listed separately.

In [22]:
sql = """-- KPI 8: payment reconciliation. Expected = sum of line_total of the order, Paid = sum of Success payments
WITH expected AS (
    SELECT order_id, SUM(line_total) AS expected_total
    FROM order_items
    GROUP BY order_id
),
paid AS (
    SELECT order_id,
           SUM(CASE WHEN payment_status = 'Success' THEN amount ELSE 0 END)           AS paid_success,
           SUM(CASE WHEN payment_status = 'Success'  THEN 1 ELSE 0 END)               AS n_success,
           COUNT(DISTINCT CASE WHEN payment_status = 'Success' THEN ROUND(amount, 2) END) AS n_distinct_success_amounts,
           SUM(CASE WHEN payment_status = 'Failed'   THEN 1 ELSE 0 END)               AS n_failed,
           SUM(CASE WHEN payment_status = 'Refunded' THEN 1 ELSE 0 END)               AS n_refunded,
           COUNT(*)                                                                   AS n_payments
    FROM payments
    GROUP BY order_id
),
joined AS (
    SELECT o.order_id, o.customer_id, o.order_date, o.order_status, o.channel,
           ROUND(e.expected_total, 2)           AS expected_total,
           ROUND(COALESCE(p.paid_success, 0), 2) AS paid_success,
           COALESCE(p.n_success, 0)                    AS n_success,
           COALESCE(p.n_distinct_success_amounts, 0)   AS n_distinct_success_amounts,
           COALESCE(p.n_failed, 0)                     AS n_failed,
           COALESCE(p.n_refunded, 0)                   AS n_refunded,
           COALESCE(p.n_payments, 0)                   AS n_payments
    FROM orders o
    LEFT JOIN expected e ON e.order_id = o.order_id
    LEFT JOIN paid p     ON p.order_id = o.order_id
)
SELECT order_id, customer_id, order_date, order_status, channel,
       expected_total, paid_success,
       ROUND(paid_success - COALESCE(expected_total, 0), 2) AS difference,
       CASE
           WHEN order_status = 'Cancelled' AND paid_success > 0 THEN 'Payment on cancelled order'
           WHEN order_status = 'Cancelled'                      THEN 'Cancelled - no payment (OK)'
           WHEN expected_total IS NULL                          THEN 'Order has no items'
           WHEN paid_success = 0 AND order_status = 'Pending'   THEN 'Pending - not paid yet (OK)'
           WHEN paid_success = 0                                THEN 'No successful payment'
           WHEN paid_success < expected_total * (1 - :pay_tolerance) THEN 'Underpaid'
           WHEN paid_success > expected_total * (1 + :pay_tolerance) THEN 'Overpaid'
           ELSE 'Matched'
       END AS recon_status,
       CASE WHEN n_success > 1 AND n_distinct_success_amounts = 1 THEN 1 ELSE 0 END AS duplicate_success_payment,
       CASE WHEN n_distinct_success_amounts > 1                   THEN 1 ELSE 0 END AS conflicting_success_amounts,
       n_success, n_failed, n_refunded, n_payments
FROM joined
ORDER BY order_id;
"""

kpi8 = q(conn, sql, PARAMS)
kpi8.head(10)

,order_id,customer_id,order_date,order_status,channel,expected_total,paid_success,difference,recon_status,duplicate_success_payment,conflicting_success_amounts,n_success,n_failed,n_refunded,n_payments
0,O00001,C0033,2022-05-10,Completed,App,"109,114.18","109,114.18",0.00,Matched,0,0,1,0,0,1
1,O00002,C0047,2024-09-21,Cancelled,Web,"21,724.64",0.00,"-21,724.64",Cancelled - no payment (OK),0,0,0,0,1,1
2,O00003,C0132,2025-01-27,Completed,Web,"33,974.88",0.00,"-33,974.88",No successful payment,0,0,0,0,0,1
3,O00004,C0144,2024-12-12,Completed,Store,"180,629.84","180,629.84",0.00,Matched,0,0,1,0,0,1
4,O00005,C0142,2025-05-08,Completed,App,"104,397.68","73,078.38","-31,319.30",Underpaid,0,0,1,0,0,1
5,O00006,C0122,2024-02-15,Completed,Store,"17,880.99","17,880.99",0.00,Matched,0,0,1,0,0,1
6,O00007,C0004,2024-03-01,Completed,Store,"57,672.52","57,672.52",0.00,Matched,0,0,1,0,0,1
7,O00008,C0118,2024-07-09,Completed,Web,"141,201.64","168,821.36","27,619.72",Overpaid,0,0,1,0,0,1
8,O00009,C0085,2025-07-09,Completed,Store,"40,778.86",0.00,"-40,778.86",No successful payment,0,0,0,0,1,1
9,O00010,C0100,2025-04-09,Completed,App,"164,403.47","168,178.45","3,774.98",Overpaid,0,0,1,0,0,1


Summary - how many orders in each status and how much money:

In [23]:
kpi8_summary = summarize_recon(kpi8)
kpi8_summary

,recon_status,orders,expected_total,paid_success
1,Matched,111,"9,772,884.77","9,772,152.88"
2,No successful payment,63,"5,435,482.92",0.00
7,Underpaid,40,"3,785,562.50","2,561,082.81"
4,Overpaid,36,"3,359,474.01","4,372,716.22"
5,Payment on cancelled order,35,"3,169,001.78","3,374,930.66"
6,Pending - not paid yet (OK),25,"2,601,591.38",0.00
0,Cancelled - no payment (OK),8,"707,619.88",0.00
3,Order has no items,2,0.00,0.00


Flags for duplicate / conflicting payments, and the 8 payments with no matching order:

In [24]:
print('Orders with duplicate Success payments  :', int(kpi8.duplicate_success_payment.sum()))
print('Orders with conflicting Success amounts :', int(kpi8.conflicting_success_amounts.sum()))
print('Orders with a Refunded payment          :', int((kpi8.n_refunded > 0).sum()))

sql = """-- KPI 8 extra: payments that were quarantined in cleaning because their order does not exist
SELECT record_key                              AS payment_id,
       reason,
       json_extract(record_json, '$.order_id') AS order_id_in_payment,
       json_extract(record_json, '$.amount')   AS raw_amount,
       json_extract(record_json, '$.payment_status') AS raw_status,
       record_json
FROM rejects
WHERE "table" = 'payments' AND reason = 'ORPHAN_ORDER'
ORDER BY record_key;
"""

kpi8_orphans = q(conn, sql, PARAMS)
kpi8_orphans.drop(columns='record_json')

Orders with duplicate Success payments  : 0
Orders with conflicting Success amounts : 0
Orders with a Refunded payment          : 29


,payment_id,reason,order_id_in_payment,raw_amount,raw_status
0,PM00026,ORPHAN_ORDER,O94535,45226.39,Refunded
1,PM00125,ORPHAN_ORDER,O93042,104461.04,Success
2,PM00156,ORPHAN_ORDER,O97748,107279.68,Success
3,PM00192,ORPHAN_ORDER,O98217,124164.45,Refunded
4,PM00264,ORPHAN_ORDER,O97158,27430.41,Success
5,PM00277,ORPHAN_ORDER,O94502,107841.66,Failed
6,PM00294,ORPHAN_ORDER,O90749,224837.1,Success
7,PM00316,ORPHAN_ORDER,O93525,70145.67,Success


**Check KPI 8**

In [25]:
n_orders_all = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
check('KPI8 one row per order', len(kpi8) == n_orders_all and kpi8.order_id.is_unique)
check('KPI8 status counts add up to all orders', kpi8_summary.orders.sum() == n_orders_all)
check('KPI8 every order has a status', kpi8.recon_status.notna().all())
check('KPI8 matched orders really are within 1%',
      (kpi8[kpi8.recon_status == 'Matched'].difference.abs() <= 0.01 * kpi8[kpi8.recon_status == 'Matched'].expected_total + 0.01).all())
check('KPI8 8 orphan payments listed', len(kpi8_orphans) == 8)
save(kpi8, 'kpi_08_payment_reconciliation')
save(kpi8_summary, 'kpi_08_payment_reconciliation_summary')
save(kpi8_orphans.drop(columns='record_json'), 'kpi_08_orphan_payments')

PASS  KPI8 one row per order
PASS  KPI8 status counts add up to all orders
PASS  KPI8 every order has a status
PASS  KPI8 matched orders really are within 1%
PASS  KPI8 8 orphan payments listed


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_08_orphan_payments.csv')

---
## Stretch KPI - Margin risk

**Question:** where did we sell below cost?

**How:** a sold line is at risk when `cost price > unit price x (1 - discount%)`. For those lines we report the products affected, units, revenue exposed and **loss = units x (cost - price after discount)**.

**Assumptions:** Completed + Returned orders; loss is calculated **before** returns (returned goods are not netted back).

In [26]:
sql = """-- Stretch: margin risk = sold lines where cost price is higher than the price after discount
SELECT p.product_id, p.product_name, p.category,
       COUNT(*)                         AS lines_affected,
       SUM(oi.quantity)                 AS units_affected,
       ROUND(SUM(oi.line_total), 2)     AS revenue_exposed,
       ROUND(SUM(oi.quantity * (p.cost_price - oi.unit_price * (1 - oi.discount_pct / 100.0))), 2) AS total_loss
FROM order_items oi
JOIN orders   o ON o.order_id   = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_status IN ('Completed', 'Returned')
  AND p.cost_price > oi.unit_price * (1 - oi.discount_pct / 100.0)
GROUP BY p.product_id, p.product_name, p.category
ORDER BY total_loss DESC;
"""

kpi9 = q(conn, sql, PARAMS)
kpi9.head(10)

,product_id,product_name,category,lines_affected,units_affected,revenue_exposed,total_loss
0,P065,Footwear Item 65,Fashion,14,44,"637,664.84","387,125.54"
1,P013,Appliances Item 13,Home & Kitchen,9,22,"308,547.93","240,142.85"
2,P060,Staples Item 60,Grocery,7,24,"257,422.93","188,407.07"
3,P039,Fitness Item 39,Sports,5,19,"99,789.68","177,181.11"
4,P001,Accessories Item 1,Electronics,5,13,"134,858.99","83,535.94"
5,P003,Staples Item 3,Grocery,9,28,"242,417.08","54,457.11"
6,P028,Appliances Item 28,Home & Kitchen,7,19,"125,241.49","54,452.52"
7,P020,Beverages Item 20,Grocery,8,25,"83,231.65","46,062.85"
8,P006,Furniture Item 6,Home & Kitchen,4,11,"30,205.60","19,843.63"
9,P051,Footwear Item 51,Fashion,3,6,"8,336.15","19,112.95"


**Check and totals**

In [27]:
check('Stretch every row has a positive loss', (kpi9.total_loss > 0).all())
totals = pd.DataFrame({
    'products_affected': [len(kpi9)],
    'lines_affected': [kpi9.lines_affected.sum()],
    'units_affected': [kpi9.units_affected.sum()],
    'revenue_exposed': [kpi9.revenue_exposed.sum()],
    'total_loss': [kpi9.total_loss.sum()],
})
display(totals)
save(kpi9, 'kpi_09_margin_risk_by_product')
save(totals, 'kpi_09_margin_risk_total')

PASS  Stretch every row has a positive loss


,products_affected,lines_affected,units_affected,revenue_exposed,total_loss
0,27,127,397,"3,324,959.26","1,369,300.57"


WindowsPath('C:/Users/acer/OneDrive/Desktop/demo_hcl/DataCore-Solution/HCL_Hackathon/kpi_outputs/kpi_09_margin_risk_total.csv')

---
## Done - output files

In [28]:
pd.DataFrame({'file': [p.name for p in sorted(OUT_DIR.glob('*.csv'))],
              'rows': [len(pd.read_csv(p)) for p in sorted(OUT_DIR.glob('*.csv'))]})

,file,rows
0,kpi_01_net_revenue_by_category_month.csv,118
1,kpi_02_top10_customers_ltv.csv,10
2,kpi_03_segment_profile.csv,4
3,kpi_04_channel_mom_growth.csv,189
4,kpi_05_product_return_rate.csv,20
5,kpi_06_supplier_scorecard.csv,19
6,kpi_07_repeat_purchase_detail.csv,98
7,kpi_07_repeat_purchase_rate.csv,1
8,kpi_08_orphan_payments.csv,8
9,kpi_08_payment_reconciliation.csv,320
